# 01 非AIのベースライン — 簡単な方法はどこで失敗するか

## ゴール
- AIを使わない**簡単な検知方法**を作り、どの異常は見つけられて、どの異常は見つけられないかを知る。
- 「**検知率が高い**」という数字が、偶然でも出てしまうことを理解する。
- 次の Phase2 以降（天気を考慮する手法）が**何を改善すべきか**を明確にする。

## 前のNotebookとのつながり
00 で、異常には4種類あり、発電量が少ないだけでは異常と言えないことを確認しました。ここでは、**天気を考えない**簡単な方法で検知してみます。

## やること
1. 素朴な固定ルール（3kWh未満なら警報）を試す
2. 正常データ（train）だけを使ってしきい値を決める検知器を3つ作る
3. 10本の系列で評価し、異常の種類ごとの検知率を見る
4. 「偶然でも検知に数えられる確率」と比べる

## キーワード
- **異常スコア**: 異常っぽさを表す数字。大きいほど異常。
- **しきい値**: スコアがこれを超えたら警報を出す境目。
- **誤警報**: 正常な日に出してしまった警報。多いと、警報が信用されなくなる。
- **適合率**: 警報を出した日のうち、本当に異常だった割合。
- **検知率（イベント単位）**: 異常イベント（例: 3日間の停電）のうち、期間中に1回でも警報が出た割合。
- **検知までの日数**: 異常が始まってから最初の警報までの日数。
- **偶然の水準**: 警報がでたらめに出ても、検知に数えられてしまう確率。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from anomaly_synth import make_split
from anomaly_eval import alarm_by_normal_quantile, false_alarms_per_month
from baselines import DETECTOR_COLUMNS, score_low_value
from run_phase1 import build_detectors, RATE

plt.rcParams["font.family"] = ["Hiragino Sans", "AppleGothic", "DejaVu Sans"]
train = make_split("train")          # 正常のみ。しきい値を決めるために使う
val = make_split("val", 1)           # 異常あり。評価用（index=0 は使わない）
x = val[DETECTOR_COLUMNS]            # 検知器が見てよい列だけ。答え（is_anomaly など）は渡さない
print(x.columns.tolist())

**何をしているか**: データを用意します。`x` は検知器に渡す入力で、**答えの列を含みません**。答えは、評価のときだけ使います。

**出力の見方**: `date, weather_class, sunshine_ratio, gsr, gen_kwh` の5列だけです。ここに `is_anomaly` などが含まれていないことを、毎回確認しましょう。

In [ ]:
alarm = score_low_value(x).to_numpy() > -3.0       # 発電量が3kWh未満なら警報
normal = (val["is_anomaly"] == 0).to_numpy()
rain = (val["weather_class"] == 2).to_numpy()

print("警報の日数:", alarm.sum())
print("  うち正常日（誤警報）:", (alarm & normal).sum(), "／ そのうち雨の日:", (alarm & normal & rain).sum())
print("  うち異常日:", (alarm & ~normal).sum(), "／ 異常日の合計:", (~normal).sum())
print("雨の正常日:", (normal & rain).sum(), "日")

**何をしているか**: 「3kWh未満なら警報」という、最も素朴なルールを試します。

**出力の見方**:
- 警報75日のうち、**正常日の誤警報が48日**。そのうち46日は雨の日です。
- 異常日71日のうち、警報が出たのは27日だけです。
- つまり、このルールは**雨を異常と間違える**のが主な問題です。警報の約6割（48/75）が誤りでは、誰も警報を信用しなくなります。
- ここから、**「天気で説明できる低下」を異常にしてはいけない**ことが分かります。

In [ ]:
detectors = build_detectors(train)
rows = []
for name, fn in detectors.items():
    thr = alarm_by_normal_quantile(fn(train[DETECTOR_COLUMNS]), RATE)   # 正常データのスコアだけで決める
    score = fn(x).to_numpy()
    al = score > thr
    rows.append(dict(検知器=name, しきい値=round(thr, 3), 警報日数=int(al.sum()),
                     誤警報_月=round(false_alarms_per_month(val, al), 2),
                     適合率=round((al & ~normal).sum() / max(1, al.sum()), 2)))
pd.DataFrame(rows)

**何をしているか**: 3つの検知器と1つのルールについて、**train（正常のみ）のスコアの上位3%**をしきい値にします。
「正常な日に警報が出る割合を約3%（約1か月に1回）に抑える」という設計です。そのうえで、val の1系列で警報を出します。

**出力の見方**:
- 3つの方法は、どれも誤警報が月1回前後に収まっています（0.6〜1.2回）。素朴ルールの誤警報より、ずっと少ないです。
- ただし**適合率は0.45〜0.6**で、警報の半分近くは誤りです。
- 固着ルール（同じ値が続く）は誤警報が0で、適合率1.0です。ただし、見つけるのは固着だけです。
- 1系列だけでは、たまたまの影響が大きいので、次で10系列に広げます。

In [ ]:
import run_phase1
run_phase1.main()

**何をしているか**: val の10系列（index=1〜10）で、全検知器を評価します（異常イベントは全部で、停電40・影20・固着20・劣化10）。

**出力の見方**:
1. **検知器ごと**: 固着ルール以外の3つは、誤警報が1か月に1回前後（0.7〜1.0）。素朴ルールだけ4.55回と、約5倍です。`auprc` は、しきい値を動かしたときの総合的な良さです。この3つはどれも0.3台で、大差ありません（標準偏差0.03〜0.05の範囲）。
2. **検知率（イベント単位）**:
   - 停電は、固着ルール以外すべて1.0で見つかります。**易しい異常**です。
   - 固着は、固着ルールだけが1.0で、他は0.05。**値の大きさを見る方法では見つかりません**。
   - 影は0.3、劣化は0.7〜0.9。
3. **日単位の検知率**: 異常日のうち警報が出た割合です。影は0.06〜0.07、劣化は0.04〜0.06。誤警報率の3%とほとんど変わりません。
4. **偶然の水準**: 警報がでたらめに出ても、劣化は64%、影は14%、固着は13%の確率で「検知」に数えられます。
   劣化の検知率0.7〜0.9は、**偶然の0.64と大差がありません**。検知までの日数が約17日と長いのも、偶然に近い証拠です。

つまり、**劣化と影は、天気を考えない方法ではほぼ見つけられていません**。長い異常は、1回でも警報が出れば「検知」になるので、数字が良く見えてしまいます。

## 試してみよう（演習）
1. **誤警報の目標を変える**: `run_phase1.py` の `RATE = 0.03` を `0.10` に変えて、もう一度 `run_phase1.main()` を実行してください（`import run_phase1` の後に `import importlib; importlib.reload(run_phase1)` が必要です）。
   見るポイントは、誤警報が約3倍に増える代わりに、**影と劣化の検知率がどれだけ上がるか**です。分かるのは、検知率は誤警報と引き換えで、しきい値を下げて見かけの検知率を上げるのは簡単だということです。
2. **移動平均の窓を変える**: `src/baselines.py` の `score_moving_average_drop` の `window=7` を `3` と `30` にして比べてください。（`src/baselines.py` を書き換えたら、カーネルを再起動してから最初のセルを実行し直します。）見るポイントは、停電・影・劣化それぞれの検知率の変化です。分かるのは、短い窓は急な落ち込みに、長い窓は緩やかな変化に向くのかどうかです。
3. **評価系列を変える**: `EVAL_INDEXES = range(1, 11)` を `range(11, 21)` にして、結果が似ているか確かめてください。分かるのは、1系列の結果はぶれるが、10系列なら安定するかどうかです。

## まとめ
- 天気を考えない方法は、**雨を異常と間違える**か、**異常を見落とす**かのどちらかになる。
- 停電は易しく、**影・劣化は天気のばらつきに埋もれて**、ほとんど見つからない。固着は値の大きさでは見つからず、専用のルールが要る。
- イベント単位の検知率は、長い異常ほど**偶然でも高く出る**。偶然の水準と、日単位の検知率も必ず見る。
- 次の Phase2 では、**日射量から期待される発電量**を予測し、そこからの外れ（残差）で検知します。天気で説明できる低下を異常と間違えなくなるはずです。